# 5. Payer & Utilization Intelligence
## Insurance Plan Performance and Member Attribution Analytics

Strategy: LIMIT 1000 on BigQuery reads -> Local CSV -> Pandas -> Spark

## Available Tables:
- **OMOP**: 24 tables
- **Medicare**: 6 tables
- **Dual Enrollment**: 1 table (SDOH)
- **CMS Codes**: 3 tables

## Pipeline Architecture:
- **Bronze**: 10 tables (payer, utilization, cost data)
- **Silver**: 6 intermediate layers (plan aggregations)
- **Gold**: 15 vertical layers -> 4 final metrics

## Final Metrics:
1. **Risk-Adjusted Utilization Rate** - HCC-normalized service usage
2. **Benefit Design Effectiveness** - Coverage optimization score
3. **High-Value Care Penetration** - Quality service adoption rate
4. **Member Attribution Stability** - Continuity of coverage index

In [21]:
!pip install google-cloud-bigquery
!pip install pandas
!pip install networkx


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [22]:
from google.cloud import bigquery
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import *
import os
from datetime import datetime
import networkx as nx
import json
import requests 

In [23]:
print("Initializing Spark...")
spark = (
    SparkSession.builder
    .appName("PayerUtilizationIntelligence")
    .master("local[*]") 
    
    # OpenLineage Configuration
    .config("spark.jars.packages", "io.openlineage:openlineage-spark_2.12:1.18.0")
    .config("spark.extraListeners", "io.openlineage.spark.agent.OpenLineageSparkListener")
    .config("spark.openlineage.transport.type", "http")
    .config("spark.openlineage.transport.url", "http://localhost:4601")
    .config("spark.openlineage.namespace", "payer_utilization")

    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8") 
    .config("spark.driver.maxResultSize", "2g")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"✅ OpenLineage → http://localhost:4601")
print(f"✅ Namespace: payer_utilization")

Initializing Spark...
Spark version: 3.5.1
Spark UI: http://mac:4040
✅ OpenLineage → http://localhost:4601
✅ Namespace: payer_utilization


In [24]:
# Configuration
PROJECT_ID = "opportune-ruler-447319-b3"
LIMIT = 1000

# Local data directory
LOCAL_DATA_DIR = "./5_data"
# os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# BigQuery datasets
DATASETS = {
    'omop': 'bigquery-public-data.cms_synthetic_patient_data_omop',
    'medicare': 'bigquery-public-data.cms_medicare',
    'dual_enrollment': 'bigquery-public-data.sdoh_cms_dual_eligible_enrollment',
    'cms_codes': 'bigquery-public-data.cms_codes'
}

print(f"\nConfiguration:")
print(f"  GCP Project: {GCP_PROJECT}")
print(f"  Local Data: {LOCAL_DATA_DIR}")
print(f"  Datasets: {len(DATASETS)}")


Configuration:
  GCP Project: opportune-ruler-447319-b3
  Local Data: ./5_data
  Datasets: 4


# STEP 1: Download Limited Data from BigQuery (LIMIT 1000)

In [25]:
def download_table_limited(dataset_name, table_name, limit=1000):
    """
    Download limited rows from BigQuery table to local CSV
    """
    full_table = f"{DATASETS[dataset_name]}.{table_name}"
    csv_path = f"{LOCAL_DATA_DIR}/{table_name}.csv"
    
    if os.path.exists(csv_path):
        print(f"  Skip {table_name} (already exists)")
        return csv_path
    
    try:
        query = f"SELECT * FROM `{full_table}` LIMIT {limit}"
        df_pd = BQ_CLIENT.query(query).to_dataframe()
        df_pd.to_csv(csv_path, index=False)
        print(f"  Downloaded {table_name}: {len(df_pd)} rows -> {csv_path}")
        return csv_path
    except Exception as e:
        print(f"  Error downloading {table_name}: {e}")
        return None

In [26]:
print("\n" + "="*80)
print("DOWNLOADING TABLES (LIMIT 1000 each)")
print("="*80)

tables_to_download = [
    ('omop', 'person'),
    ('omop', 'observation_period'),
    ('omop', 'payer_plan_period'),
    ('omop', 'condition_occurrence'),
    ('omop', 'procedure_occurrence'),
    ('omop', 'drug_exposure'),
    ('omop', 'cost'),
    ('omop', 'care_site'),
    ('omop', 'provider'),
    ('medicare', 'inpatient_charges_2011')
]

downloaded_files = {}
for dataset, table in tables_to_download:
    path = download_table_limited(dataset, table, limit=1000)
    if path:
        downloaded_files[table] = path

print(f"\nDownloaded {len(downloaded_files)} tables")


DOWNLOADING TABLES (LIMIT 1000 each)
  Skip person (already exists)
  Skip observation_period (already exists)
  Skip payer_plan_period (already exists)
  Skip condition_occurrence (already exists)
  Skip procedure_occurrence (already exists)
  Skip drug_exposure (already exists)
  Skip cost (already exists)
  Skip care_site (already exists)
  Skip provider (already exists)
  Skip inpatient_charges_2011 (already exists)

Downloaded 10 tables


# STEP 2: Load CSV to Pandas then Spark (Bronze Layer)

In [27]:
def load_csv_to_spark_with_lineage(dataset_key, table_name, layer="bronze"):
    """Load CSV file into Spark - NO metadata columns, NO count()"""
    try:
        csv_path = f"{LOCAL_DATA_DIR}/{table_name}.csv"
        
        if not os.path.exists(csv_path):
            print(f"  ✗ {dataset_key}.{table_name}: CSV not found at {csv_path}")
            return None
        
        # Read CSV - NO metadata, NO count()
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(csv_path)
        
        print(f"  ✓ {dataset_key}.{table_name}: loaded into {layer} layer")
        return df
        
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: Error - {str(e)}")
        return None


def add_layer_metadata(df, layer, source_tables, target_table=None):
    """DEPRECATED - Don't use metadata columns"""
    return df  # Just return DataFrame as-is

In [28]:
print("\n" + "="*80)
print("BRONZE LAYER: Load CSV -> Pandas -> Spark")
print("="*80)

os.makedirs("./output/bronze", exist_ok=True)

def load_and_write_bronze(dataset_key, table_name):
    """Load CSV and write immediately to capture lineage"""
    df = load_csv_to_spark_with_lineage(dataset_key, table_name, "bronze")
    if df is not None:
        output_path = f"./output/bronze/bronze_{dataset_key}_{table_name}"
        df.write.mode("overwrite").parquet(output_path)
        print(f"    → Written to bronze")
        # Read back for use in Silver
        return spark.read.parquet(output_path)
    return None


bronze_person = load_and_write_bronze("omop", "person")
bronze_obs_period = load_and_write_bronze("omop", "observation_period")
bronze_payer = load_and_write_bronze("omop", "payer_plan_period")
bronze_condition = load_and_write_bronze("omop", "condition_occurrence")
bronze_procedure = load_and_write_bronze("omop", "procedure_occurrence")
bronze_drug = load_and_write_bronze("omop", "drug_exposure")
bronze_cost = load_and_write_bronze("omop", "cost")
bronze_care_site = load_and_write_bronze("omop", "care_site")
bronze_provider = load_and_write_bronze("omop", "provider")
bronze_inpatient = load_and_write_bronze("medicare", "inpatient_charges_2011")

print("\n✓ Bronze layer loaded (10 tables)")


BRONZE LAYER: Load CSV -> Pandas -> Spark
  ✓ omop.person: loaded into bronze layer
    → Written to bronze
  ✓ omop.observation_period: loaded into bronze layer


25/11/23 23:07:51 ERROR ContextFactory: Query execution is null: can't emit event for executionId 21
25/11/23 23:07:51 ERROR ContextFactory: Query execution is null: can't emit event for executionId 21


    → Written to bronze
  ✓ omop.payer_plan_period: loaded into bronze layer
    → Written to bronze


25/11/23 23:07:51 ERROR ContextFactory: Query execution is null: can't emit event for executionId 24
25/11/23 23:07:51 ERROR ContextFactory: Query execution is null: can't emit event for executionId 24


  ✓ omop.condition_occurrence: loaded into bronze layer
    → Written to bronze


...

  ✓ omop.cost: loaded into bronze layer
    → Written to bronze
  ✓ omop.care_site: loaded into bronze layer


25/11/23 23:07:52 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 23:07:52 ERROR ContextFactory: Query execution is null: can't emit event for executionId 32
25/11/23 23:07:52 ERROR ContextFactory: Query execution is null: can't emit event for executionId 33
25/11/23 23:07:52 ERROR ContextFactory: Query execution is null: can't emit event for executionId 33


    → Written to bronze
  ✓ omop.provider: loaded into bronze layer
    → Written to bronze


25/11/23 23:07:52 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 23:07:52 ERROR ContextFactory: Query execution is null: can't emit event for executionId 36
25/11/23 23:07:53 ERROR ContextFactory: Query execution is null: can't emit event for executionId 38
25/11/23 23:07:53 ERROR ContextFactory: Query execution is null: can't emit event for executionId 38


  ✓ medicare.inpatient_charges_2011: loaded into bronze layer
    → Written to bronze

✓ Bronze layer loaded (10 tables)


# STEP 3: SILVER LAYER (6 Intermediate Transformations)

In [29]:
print("\n" + "="*80)
print("SILVER 1: Member Demographics & Enrollment")
print("="*80)

os.makedirs("./output/silver", exist_ok=True)

silver_member_enrollment = bronze_person \
    .join(bronze_obs_period, 'person_id', 'left') \
    .join(bronze_payer, 'person_id', 'left') \
    .select(
        F.col('person_id'),
        F.col('gender_concept_id'),
        F.col('birth_datetime'),
        F.col('race_concept_id'),
        F.col('ethnicity_concept_id'),
        F.col('observation_period_start_date'),
        F.col('observation_period_end_date'),
        F.col('payer_plan_period_start_date'),
        F.col('payer_plan_period_end_date'),
        F.coalesce(F.col('payer_concept_id'), F.lit(0)).alias('payer_concept_id'),
        F.coalesce(F.col('plan_concept_id'), F.lit(0)).alias('plan_concept_id')
    ) \
    .withColumn(
        'birth_datetime_converted',
        F.to_date(F.col('birth_datetime').cast('string'))
    ) \
    .withColumn(
        'payer_plan_period_start_date_converted',
        F.to_date(F.col('payer_plan_period_start_date').cast('string'))
    ) \
    .withColumn(
        'payer_plan_period_end_date_converted',
        F.to_date(F.col('payer_plan_period_end_date').cast('string'))
    ) \
    .withColumn(
        'age',
        F.floor(F.datediff(F.current_date(), F.col('birth_datetime_converted')) / 365.25)
    ) \
    .withColumn(
        'enrollment_days',
        F.datediff(
            F.coalesce(F.col('payer_plan_period_end_date_converted'), F.current_date()),
            F.col('payer_plan_period_start_date_converted')
        )
    ) \
    .drop('birth_datetime', 'payer_plan_period_start_date', 'payer_plan_period_end_date') \
    .withColumnRenamed('birth_datetime_converted', 'birth_datetime') \
    .withColumnRenamed('payer_plan_period_start_date_converted', 'payer_plan_period_start_date') \
    .withColumnRenamed('payer_plan_period_end_date_converted', 'payer_plan_period_end_date')

# Write immediately!
silver_member_enrollment.write.mode("overwrite").parquet("./output/silver/silver_member_enrollment")
print(f"✓ ME written")

# Read back
silver_member_enrollment = spark.read.parquet("./output/silver/silver_member_enrollment")
print(f"  ME: {silver_member_enrollment.count()}")

print(f"Silver 1 complete: {silver_member_enrollment.count()} members")


SILVER 1: Member Demographics & Enrollment
✓ ME written
  ME: 1000
Silver 1 complete: 1000 members


25/11/23 23:07:53 ERROR ContextFactory: Query execution is null: can't emit event for executionId 41
25/11/23 23:07:53 ERROR ContextFactory: Query execution is null: can't emit event for executionId 41
25/11/23 23:07:53 ERROR ContextFactory: Query execution is null: can't emit event for executionId 41


In [30]:
print("\n" + "="*80)
print("SILVER 2: Service Utilization by Member")
print("="*80)

condition_util = bronze_condition.groupBy('person_id').agg(
    F.count('*').alias('condition_count'),
    F.countDistinct('condition_concept_id').alias('unique_conditions')
)

procedure_util = bronze_procedure.groupBy('person_id').agg(
    F.count('*').alias('procedure_count'),
    F.countDistinct('procedure_concept_id').alias('unique_procedures')
)

drug_util = bronze_drug.groupBy('person_id').agg(
    F.count('*').alias('drug_count'),
    F.countDistinct('drug_concept_id').alias('unique_drugs'),
    F.sum('days_supply').alias('total_days_supply')
)

silver_utilization = silver_member_enrollment.select('person_id') \
    .join(condition_util, 'person_id', 'left') \
    .join(procedure_util, 'person_id', 'left') \
    .join(drug_util, 'person_id', 'left') \
    .fillna(0)

# Write immediately!
silver_utilization.write.mode("overwrite").parquet("./output/silver/silver_utilization")
print(f"✓ Utilization written")

# Read back
silver_utilization = spark.read.parquet("./output/silver/silver_utilization")
print(f"  Utilization: {silver_utilization.count()}")

print(f"Silver 2 complete: {silver_utilization.count()} records")


SILVER 2: Service Utilization by Member
✓ Utilization written
  Utilization: 1000
Silver 2 complete: 1000 records


In [31]:
print("\n" + "="*80)
print("SILVER 3: Cost Attribution by Payer")
print("="*80)

payer_mapping = bronze_payer.select(
    F.col('payer_plan_period_id').alias('payer_period_id'),
    'person_id', 
    'payer_concept_id'
)

silver_cost_by_payer = bronze_cost \
    .join(payer_mapping, 
          bronze_cost.payer_plan_period_id == payer_mapping.payer_period_id, 
          'left') \
    .groupBy('person_id', 'payer_concept_id').agg(
        F.sum('total_paid').alias('total_paid_by_payer'),
        F.sum('paid_by_payer').alias('payer_paid'),
        F.sum('paid_by_patient').alias('patient_paid'),
        F.count('*').alias('cost_event_count')
    )

# Write immediately!
silver_cost_by_payer.write.mode("overwrite").parquet("./output/silver/silver_cost_by_payer")
print(f"✓ CP written")

# Read back
silver_cost_by_payer = spark.read.parquet("./output/silver/silver_cost_by_payer")
print(f"  CP: {silver_cost_by_payer.count()}")

print(f"Silver 3 complete: {silver_cost_by_payer.count()} payer-member combinations")


SILVER 3: Cost Attribution by Payer
✓ CP written
  CP: 1
Silver 3 complete: 1 payer-member combinations


In [32]:
print("\n" + "="*80)
print("SILVER 4: Provider Network Attribution")
print("="*80)

member_provider = bronze_condition \
    .join(bronze_provider, bronze_condition.provider_id == bronze_provider.provider_id, 'left') \
    .groupBy('person_id', bronze_provider.provider_id).agg(
        F.count('*').alias('visit_count')
    )

w = W.partitionBy('person_id').orderBy(F.desc('visit_count'))
silver_provider_attribution = member_provider \
    .withColumn('rank', F.row_number().over(w)) \
    .filter(F.col('rank') == 1) \
    .select('person_id', F.col('provider_id').alias('attributed_provider_id'), 'visit_count')

# Write immediately!
silver_provider_attribution.write.mode("overwrite").parquet("./output/silver/silver_provider_attribution")
print(f"✓ PA written")

# Read back
silver_provider_attribution = spark.read.parquet("./output/silver/silver_provider_attribution")
print(f"  PA: {silver_provider_attribution.count()}")

print(f"Silver 4 complete: {silver_provider_attribution.count()} attributions")


SILVER 4: Provider Network Attribution
✓ PA written
  PA: 1000
Silver 4 complete: 1000 attributions


25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 44
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 44
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 44
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 45
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 45
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 45
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 46
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 46
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 46
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for execu

In [33]:
print("\n" + "="*80)
print("SILVER 5: Risk Score Calculation (HCC-like)")
print("="*80)

silver_risk_scores = silver_member_enrollment \
    .join(silver_utilization.select('person_id', 'unique_conditions'), 'person_id', 'left') \
    .withColumn(
        'age_risk_factor',
        F.when(F.col('age') >= 85, 2.5)
         .when(F.col('age') >= 75, 2.0)
         .when(F.col('age') >= 65, 1.5)
         .when(F.col('age') >= 50, 1.2)
         .otherwise(1.0)
    ) \
    .withColumn(
        'condition_risk_factor',
        F.when(F.col('unique_conditions') >= 10, 2.0)
         .when(F.col('unique_conditions') >= 5, 1.5)
         .when(F.col('unique_conditions') >= 2, 1.2)
         .otherwise(1.0)
    ) \
    .withColumn(
        'hcc_risk_score',
        F.col('age_risk_factor') * F.col('condition_risk_factor')
    ) \
    .select('person_id', 'hcc_risk_score', 'age_risk_factor', 'condition_risk_factor')

# Write immediately!
silver_risk_scores.write.mode("overwrite").parquet("./output/silver/silver_risk_scores")
print(f"✓ RS written")

# Read back
silver_risk_scores = spark.read.parquet("./output/silver/silver_risk_scores")
print(f"  RS: {silver_risk_scores.count()}")

print(f"Silver 5 complete: {silver_risk_scores.count()} risk scores")


SILVER 5: Risk Score Calculation (HCC-like)
✓ RS written
  RS: 1000
Silver 5 complete: 1000 risk scores


In [34]:
print("\n" + "="*80)
print("SILVER 6: Plan Benefit Design Features")
print("="*80)

plan_agg = bronze_payer.groupBy('payer_concept_id', 'plan_concept_id').agg(
    F.count('person_id').alias('member_count'),
    F.countDistinct('person_id').alias('unique_members')
)

cost_agg = silver_cost_by_payer.groupBy('payer_concept_id').agg(
    F.avg('total_paid_by_payer').alias('avg_total_cost'),
    F.avg('payer_paid').alias('avg_payer_share'),
    F.avg('patient_paid').alias('avg_patient_share')
)

silver_plan_features = plan_agg \
    .join(cost_agg, 'payer_concept_id', 'left') \
    .withColumn(
        'payer_cost_share_pct',
        F.col('avg_payer_share') / F.greatest(F.col('avg_total_cost'), F.lit(1)) * 100
    )

# Write immediately!
silver_plan_features.write.mode("overwrite").parquet("./output/silver/silver_plan_features")
print(f"✓ PF written")

# Read back
silver_plan_features = spark.read.parquet("./output/silver/silver_plan_features")
print(f"  PF: {silver_risk_scores.count()}")

print(f"Silver 6 complete: {silver_plan_features.count()} plan features")


SILVER 6: Plan Benefit Design Features
✓ PF written
  PF: 1000
Silver 6 complete: 1 plan features


25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 56
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 56
25/11/23 23:07:55 ERROR ContextFactory: Query execution is null: can't emit event for executionId 56


# STEP 4: GOLD LAYER (15 Vertical Transformations)

In [35]:
print("\n" + "="*80)
print("GOLD LEVEL 1: Integrated Member Profile")
print("="*80)

os.makedirs("./output/gold", exist_ok=True)

# Clean up each source first
member_base = silver_member_enrollment.select(
    'person_id',
    'gender_concept_id',
    'birth_datetime',
    'race_concept_id',
    'ethnicity_concept_id',
    'observation_period_start_date',
    'observation_period_end_date',
    'payer_plan_period_start_date',
    'payer_plan_period_end_date',
    'payer_concept_id',
    'plan_concept_id',
    'age',
    'enrollment_days'
)

gold_l1_member_profile = member_base \
    .join(silver_utilization, 'person_id', 'left') \
    .join(
        silver_cost_by_payer.select(
            F.col('person_id').alias('cost_pid'),
            F.col('payer_concept_id').alias('cost_payer_id'),
            'total_paid_by_payer',
            'payer_paid',
            'patient_paid',
            'cost_event_count'
        ),
        member_base.person_id == F.col('cost_pid'),
        'left'
    ) \
    .drop('cost_pid') \
    .withColumn(
        'payer_concept_id_final',
        F.coalesce(F.col('payer_concept_id'), F.col('cost_payer_id'))
    ) \
    .drop('payer_concept_id', 'cost_payer_id') \
    .withColumnRenamed('payer_concept_id_final', 'payer_concept_id') \
    .join(silver_provider_attribution, 'person_id', 'left') \
    .join(silver_risk_scores, 'person_id', 'left') \
    .fillna(0)

# Write immediately!
gold_l1_member_profile.write.mode("overwrite").parquet("./output/gold/gold_l1_member_profile")
print(f"✓ l1MP written")

# Read back
gold_l1_member_profile = spark.read.parquet("./output/gold/gold_l1_member_profile")
print(f"  l1MP: {gold_l1_member_profile.count()}")

print(f"Gold L1 complete: {gold_l1_member_profile.count()} integrated profiles")


GOLD LEVEL 1: Integrated Member Profile


25/11/23 23:07:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✓ l1MP written
  l1MP: 1000
Gold L1 complete: 1000 integrated profiles


In [36]:
print("\n" + "="*80)
print("GOLD LEVEL 2: Utilization Density Metrics")
print("="*80)

gold_l2_utilization_density = gold_l1_member_profile \
    .withColumn(
        'enrollment_years',
        F.greatest(F.col('enrollment_days') / 365.25, F.lit(0.5))
    ) \
    .withColumn(
        'annual_condition_rate',
        F.col('condition_count') / F.col('enrollment_years')
    ) \
    .withColumn(
        'annual_procedure_rate',
        F.col('procedure_count') / F.col('enrollment_years')
    ) \
    .withColumn(
        'annual_drug_rate',
        F.col('drug_count') / F.col('enrollment_years')
    ) \
    .withColumn(
        'total_annual_utilization',
        F.col('annual_condition_rate') + F.col('annual_procedure_rate') + F.col('annual_drug_rate')
    )

# Write immediately!
gold_l2_utilization_density.write.mode("overwrite").parquet("./output/gold/gold_l2_utilization_density")
print(f"✓ l2UD written")

# Read back
gold_l2_utilization_density = spark.read.parquet("./output/gold/gold_l2_utilization_density")
print(f"  l2UD: {gold_l2_utilization_density.count()}")

print("Gold L2 complete")


GOLD LEVEL 2: Utilization Density Metrics
✓ l2UD written
  l2UD: 1000
Gold L2 complete


In [37]:
print("\n" + "="*80)
print("GOLD LEVEL 3: Risk-Adjusted Utilization")
print("="*80)

gold_l3_risk_adjusted = gold_l2_utilization_density \
    .withColumn(
        'expected_utilization',
        F.col('hcc_risk_score') * 10.0
    ) \
    .withColumn(
        'utilization_vs_expected',
        F.col('total_annual_utilization') / F.greatest(F.col('expected_utilization'), F.lit(1))
    ) \
    .withColumn(
        'risk_adjusted_utilization_rate',
        F.col('total_annual_utilization') / F.greatest(F.col('hcc_risk_score'), F.lit(1))
    )

# Write immediately!
gold_l3_risk_adjusted.write.mode("overwrite").parquet("./output/gold/gold_l3_risk_adjusted")
print(f"✓ l3RA written")

# Read back
gold_l3_risk_adjusted = spark.read.parquet("./output/gold/gold_l3_risk_adjusted")
print(f"  l3RA: {gold_l3_risk_adjusted.count()}")

print("Gold L3 complete")


GOLD LEVEL 3: Risk-Adjusted Utilization
✓ l3RA written
  l3RA: 1000
Gold L3 complete


In [38]:
print("\n" + "="*80)
print("GOLD LEVEL 4: Cost per Member per Month (PMPM)")
print("="*80)

gold_l4_pmpm = gold_l3_risk_adjusted \
    .withColumn(
        'enrollment_months',
        F.greatest(F.col('enrollment_days') / 30.0, F.lit(1))
    ) \
    .withColumn(
        'total_cost_pmpm',
        F.col('total_paid_by_payer') / F.col('enrollment_months')
    ) \
    .withColumn(
        'payer_cost_pmpm',
        F.col('payer_paid') / F.col('enrollment_months')
    ) \
    .withColumn(
        'patient_cost_pmpm',
        F.col('patient_paid') / F.col('enrollment_months')
    )

# Write immediately!
gold_l4_pmpm.write.mode("overwrite").parquet("./output/gold/gold_l4_pmpm")
print(f"✓ l4pmpm written")

# Read back
gold_l4_pmpm = spark.read.parquet("./output/gold/gold_l4_pmpm")
print(f"  l4pmpm: {gold_l4_pmpm.count()}")

print("Gold L4 complete")


GOLD LEVEL 4: Cost per Member per Month (PMPM)
✓ l4pmpm written
  l4pmpm: 1000
Gold L4 complete


25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 59
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 59
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 59
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 60
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 60
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 60
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 61
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 61
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 62
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for execu

In [39]:
print("\n" + "="*80)
print("GOLD LEVEL 5: Risk-Adjusted PMPM")
print("="*80)

gold_l5_risk_adjusted_pmpm = gold_l4_pmpm \
    .withColumn(
        'expected_pmpm',
        F.col('hcc_risk_score') * 500.0
    ) \
    .withColumn(
        'pmpm_efficiency_ratio',
        F.col('total_cost_pmpm') / F.greatest(F.col('expected_pmpm'), F.lit(100))
    ) \
    .withColumn(
        'risk_adjusted_pmpm',
        F.col('total_cost_pmpm') / F.greatest(F.col('hcc_risk_score'), F.lit(1))
    )

# Write immediately!
gold_l5_risk_adjusted_pmpm.write.mode("overwrite").parquet("./output/gold/gold_l5_risk_adjusted_pmpm")
print(f"✓ l5RARApmpm written")

# Read back
gold_l5_risk_adjusted_pmpm = spark.read.parquet("./output/gold/gold_l5_risk_adjusted_pmpm")
print(f"  l5RApmpm: {gold_l5_risk_adjusted_pmpm.count()}")

print("Gold L5 complete")


GOLD LEVEL 5: Risk-Adjusted PMPM


25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 64
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 64
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 64
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 65
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 65


✓ l5RARApmpm written


25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 66
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 66
25/11/23 23:07:56 ERROR ContextFactory: Query execution is null: can't emit event for executionId 66


  l5RApmpm: 1000
Gold L5 complete


In [40]:
print("\n" + "="*80)
print("GOLD LEVEL 6: Plan Coverage Effectiveness")
print("="*80)

plan_cost_share = silver_plan_features.select(
    F.col('payer_concept_id').alias('plan_payer_id'),
    'payer_cost_share_pct'
)

gold_l6_coverage = gold_l5_risk_adjusted_pmpm \
    .join(plan_cost_share, 
          gold_l5_risk_adjusted_pmpm.payer_concept_id == plan_cost_share.plan_payer_id, 
          'left') \
    .drop('plan_payer_id') \
    .withColumn(
        'patient_burden_ratio',
        F.col('patient_paid') / F.greatest(F.col('total_paid_by_payer'), F.lit(1))
    ) \
    .withColumn(
        'coverage_adequacy_score',
        100 * (1 - F.least(F.col('patient_burden_ratio'), F.lit(1)))
    )

# Write immediately!
gold_l6_coverage.write.mode("overwrite").parquet("./output/gold/gold_l6_coverage")
print(f"✓ l6coverage written")

# Read back
gold_l6_coverage = spark.read.parquet("./output/gold/gold_l6_coverage")
print(f"  l6coverage: {gold_l6_coverage.count()}")

print("Gold L6 complete")


GOLD LEVEL 6: Plan Coverage Effectiveness
✓ l6coverage written
  l6coverage: 1000
Gold L6 complete


25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 68
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 68
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 68


In [41]:
print("\n" + "="*80)
print("GOLD LEVEL 7: High-Value Service Utilization")
print("="*80)

gold_l7_high_value = gold_l6_coverage \
    .withColumn(
        'preventive_procedure_ratio',
        F.when(F.col('procedure_count') > 0, 
               F.lit(0.3) * F.col('procedure_count') / F.col('procedure_count'))
         .otherwise(0)
    ) \
    .withColumn(
        'chronic_drug_adherence_proxy',
        F.when(F.col('unique_conditions') >= 2,
               F.least(F.col('total_days_supply') / (F.col('enrollment_days') * F.col('unique_conditions')), F.lit(1)))
         .otherwise(0)
    ) \
    .withColumn(
        'high_value_care_score',
        (F.col('preventive_procedure_ratio') * 50) + (F.col('chronic_drug_adherence_proxy') * 50)
    )

# Write immediately!
gold_l7_high_value.write.mode("overwrite").parquet("./output/gold/gold_l7_high_value")
print(f"✓ l7HV written")

# Read back
gold_l7_high_value = spark.read.parquet("./output/gold/gold_l7_high_value")
print(f"  l7HV: {gold_l7_high_value.count()}")

print("Gold L7 complete")


GOLD LEVEL 7: High-Value Service Utilization
✓ l7HV written
  l7HV: 1000
Gold L7 complete


In [42]:
print("\n" + "="*80)
print("GOLD LEVEL 8: Provider Attribution Continuity")
print("="*80)

gold_l8_attribution = gold_l7_high_value \
    .withColumn(
        'has_attributed_provider',
        F.when(F.col('attributed_provider_id').isNotNull(), 1).otherwise(0)
    ) \
    .withColumn(
        'attribution_strength',
        F.when(F.col('visit_count') >= 5, 1.0)
         .when(F.col('visit_count') >= 3, 0.7)
         .when(F.col('visit_count') >= 1, 0.4)
         .otherwise(0)
    ) \
    .withColumn(
        'continuity_score',
        F.col('has_attributed_provider') * F.col('attribution_strength') * 100
    )

# Write immediately!
gold_l8_attribution.write.mode("overwrite").parquet("./output/gold/gold_l8_attribution")
print(f"✓ l8A written")

# Read back
gold_l8_attribution = spark.read.parquet("./output/gold/gold_l8_attribution")
print(f"  l8A: {gold_l8_attribution.count()}")

print("Gold L8 complete")


GOLD LEVEL 8: Provider Attribution Continuity
✓ l8A written
  l8A: 1000
Gold L8 complete


25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 70
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 70
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 70
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 71
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 71


In [43]:
print("\n" + "="*80)
print("GOLD LEVEL 9: Member Tenure & Stability")
print("="*80)

gold_l9_stability = gold_l8_attribution \
    .withColumn(
        'tenure_years',
        F.col('enrollment_days') / 365.25
    ) \
    .withColumn(
        'tenure_stability_factor',
        F.when(F.col('tenure_years') >= 5, 1.0)
         .when(F.col('tenure_years') >= 3, 0.8)
         .when(F.col('tenure_years') >= 1, 0.6)
         .otherwise(0.3)
    ) \
    .withColumn(
        'enrollment_gap_risk',
        F.when(F.col('payer_plan_period_end_date').isNull(), 0)
         .otherwise(1)
    )

# Write immediately!
gold_l9_stability.write.mode("overwrite").parquet("./output/gold/gold_l9_stability")
print(f"✓ l9S written")

# Read back
gold_l9_stability = spark.read.parquet("./output/gold/gold_l9_stability")
print(f"  l9S: {gold_l9_stability.count()}")

print("Gold L9 complete")


GOLD LEVEL 9: Member Tenure & Stability
✓ l9S written


25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 72
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 72
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 72
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 73
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 73


  l9S: 1000
Gold L9 complete


25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 74
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 74
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 74
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 75
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 75


In [44]:
print("\n" + "="*80)
print("GOLD LEVEL 10: Plan Performance Benchmarking")
print("="*80)

plan_benchmarks = gold_l9_stability \
    .groupBy('payer_concept_id').agg(
        F.mean('total_cost_pmpm').alias('plan_avg_pmpm'),
        F.mean('risk_adjusted_pmpm').alias('plan_avg_risk_adj_pmpm'),
        F.mean('coverage_adequacy_score').alias('plan_avg_coverage'),
        F.count('*').alias('plan_member_count')
    ).select(
        F.col('payer_concept_id').alias('benchmark_payer_id'),
        'plan_avg_pmpm',
        'plan_avg_risk_adj_pmpm',
        'plan_avg_coverage',
        'plan_member_count'
    )

gold_l10_benchmarked = gold_l9_stability \
    .join(plan_benchmarks, 
          gold_l9_stability.payer_concept_id == plan_benchmarks.benchmark_payer_id, 
          'left') \
    .drop('benchmark_payer_id') \
    .withColumn(
        'pmpm_vs_plan_avg',
        F.col('total_cost_pmpm') / F.greatest(F.col('plan_avg_pmpm'), F.lit(100))
    ) \
    .withColumn(
        'coverage_vs_plan_avg',
        F.col('coverage_adequacy_score') / F.greatest(F.col('plan_avg_coverage'), F.lit(50))
    )

# Write immediately!
gold_l10_benchmarked.write.mode("overwrite").parquet("./output/gold/gold_l10_benchmarked")
print(f"✓ l10B written")

# Read back
gold_l10_benchmarked = spark.read.parquet("./output/gold/gold_l10_benchmarked")
print(f"  l10B: {gold_l10_benchmarked.count()}")

print("Gold L10 complete")


GOLD LEVEL 10: Plan Performance Benchmarking


25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 76
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 76
25/11/23 23:07:57 ERROR ContextFactory: Query execution is null: can't emit event for executionId 76


✓ l10B written
  l10B: 1000
Gold L10 complete


In [45]:
print("\n" + "="*80)
print("GOLD LEVEL 11: Value-Based Care Alignment")
print("="*80)

gold_l11_vbc = gold_l10_benchmarked \
    .withColumn(
        'quality_utilization_ratio',
        F.col('high_value_care_score') / F.greatest(F.col('total_annual_utilization'), F.lit(1))
    ) \
    .withColumn(
        'cost_quality_efficiency',
        F.col('high_value_care_score') / F.greatest(F.col('risk_adjusted_pmpm'), F.lit(100))
    ) \
    .withColumn(
        'vbc_alignment_score',
        (F.col('quality_utilization_ratio') * 0.6 + F.col('cost_quality_efficiency') * 0.4) * 100
    )

# Write immediately!
gold_l11_vbc.write.mode("overwrite").parquet("./output/gold/gold_l11_vbc")
print(f"✓ l11vbc written")

# Read back
gold_l11_vbc = spark.read.parquet("./output/gold/gold_l11_vbc")
print(f"  l11vbc: {gold_l11_vbc.count()}")

print("Gold L11 complete")


GOLD LEVEL 11: Value-Based Care Alignment
✓ l11vbc written
  l11vbc: 1000
Gold L11 complete


In [46]:
print("\n" + "="*80)
print("GOLD LEVEL 12: Member Engagement Index")
print("="*80)

gold_l12_engagement = gold_l11_vbc \
    .withColumn(
        'utilization_engagement',
        F.when(F.col('total_annual_utilization') > 0, 1).otherwise(0)
    ) \
    .withColumn(
        'provider_engagement',
        F.col('has_attributed_provider')
    ) \
    .withColumn(
        'preventive_engagement',
        F.when(F.col('preventive_procedure_ratio') > 0.2, 1).otherwise(0)
    ) \
    .withColumn(
        'member_engagement_index',
        (F.col('utilization_engagement') + F.col('provider_engagement') + F.col('preventive_engagement')) / 3.0 * 100
    )

# Write immediately!
gold_l12_engagement.write.mode("overwrite").parquet("./output/gold/gold_l12_engagement")
print(f"✓ l12E written")

# Read back
gold_l12_engagement = spark.read.parquet("./output/gold/gold_l12_engagement")
print(f"  l12E: {gold_l12_engagement.count()}")

print("Gold L12 complete")


GOLD LEVEL 12: Member Engagement Index
✓ l12E written
  l12E: 1000
Gold L12 complete


In [47]:
print("\n" + "="*80)
print("GOLD LEVEL 13: METRIC 1 - Risk-Adjusted Utilization Rate")
print("="*80)

gold_l13_metric1 = gold_l12_engagement \
    .withColumn(
        'normalized_utilization',
        F.col('total_annual_utilization') / 50.0
    ) \
    .withColumn(
        'normalized_risk',
        F.col('hcc_risk_score') / 2.5
    ) \
    .withColumn(
        'risk_adjusted_utilization_rate_final',
        F.least(
            (F.col('normalized_utilization') / F.greatest(F.col('normalized_risk'), F.lit(0.5))) * 100,
            F.lit(200)
        )
    )

# Write immediately!
gold_l13_metric1.write.mode("overwrite").parquet("./output/gold/gold_l13_metric1")
print(f"✓ l13M1 written")

# Read back
gold_l13_metric1 = spark.read.parquet("./output/gold/gold_l13_metric1")
print(f"  l13M1: {gold_l13_metric1.count()}")

print("Gold L13 complete: METRIC 1 calculated")


GOLD LEVEL 13: METRIC 1 - Risk-Adjusted Utilization Rate
✓ l13M1 written
  l13M1: 1000
Gold L13 complete: METRIC 1 calculated


25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 78
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 78
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 78
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 79
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 79


In [48]:
print("\n" + "="*80)
print("GOLD LEVEL 14: METRIC 2 - Benefit Design Effectiveness")
print("="*80)

gold_l14_metric2 = gold_l13_metric1 \
    .withColumn(
        'benefit_design_effectiveness',
        (
            F.col('coverage_adequacy_score') * 0.4 +
            F.col('high_value_care_score') * 0.3 +
            (100 - F.least(F.col('patient_burden_ratio') * 100, F.lit(100))) * 0.3
        )
    )

# Write immediately!
gold_l14_metric2.write.mode("overwrite").parquet("./output/gold/gold_l14_metric2")
print(f"✓ l14M2 written")

# Read back
gold_l14_metric2 = spark.read.parquet("./output/gold/gold_l14_metric2")
print(f"  l14M2: {gold_l14_metric2.count()}")

print("Gold L14 complete: METRIC 2 calculated")

25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 80
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 80
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 80
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 81
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 81



GOLD LEVEL 14: METRIC 2 - Benefit Design Effectiveness
✓ l14M2 written


25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 82
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 82
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 82
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 83
25/11/23 23:07:58 ERROR ContextFactory: Query execution is null: can't emit event for executionId 83


  l14M2: 1000
Gold L14 complete: METRIC 2 calculated


In [49]:
print("\n" + "="*80)
print("GOLD LEVEL 15: METRICS 3 & 4")
print("="*80)

gold_l15_final_metrics = gold_l14_metric2 \
    .withColumn(
        'high_value_care_penetration',
        F.col('high_value_care_score')
    ) \
    .withColumn(
        'member_attribution_stability',
        (
            F.col('continuity_score') * 0.4 +
            F.col('tenure_stability_factor') * 30 +
            (1 - F.col('enrollment_gap_risk')) * 30
        )
    )

# Write immediately!
gold_l15_final_metrics.write.mode("overwrite").parquet("./output/gold/gold_l15_final_metrics")
print(f"✓ l15M written")

# Read back
gold_l15_final_metrics = spark.read.parquet("./output/gold/gold_l15_final_metrics")
print(f"  l15M: {gold_l15_final_metrics.count()}")

print("Gold L15 complete: All 4 metrics calculated")
print("\nFinal Metrics:")
print("  1. risk_adjusted_utilization_rate_final")
print("  2. benefit_design_effectiveness")
print("  3. high_value_care_penetration")
print("  4. member_attribution_stability")

25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 84
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 84
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 84
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 85
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 85



GOLD LEVEL 15: METRICS 3 & 4
✓ l15M written


25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 86
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 86
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 86


  l15M: 1000
Gold L15 complete: All 4 metrics calculated

Final Metrics:
  1. risk_adjusted_utilization_rate_final
  2. benefit_design_effectiveness
  3. high_value_care_penetration
  4. member_attribution_stability


# STEP 5: Final Metrics Summary & Save

In [50]:
print("\n" + "="*80)
print("FINAL METRICS SUMMARY")
print("="*80)

final_metrics = gold_l15_final_metrics.select(
    'person_id',
    'payer_concept_id',
    'risk_adjusted_utilization_rate_final',
    'benefit_design_effectiveness',
    'high_value_care_penetration',
    'member_attribution_stability',
    'hcc_risk_score',
    'total_cost_pmpm',
    'enrollment_days'
)

final_metrics.show(10, truncate=False)

agg_stats = final_metrics.agg(
    F.mean('risk_adjusted_utilization_rate_final').alias('avg_util_rate'),
    F.mean('benefit_design_effectiveness').alias('avg_benefit_eff'),
    F.mean('high_value_care_penetration').alias('avg_hv_penetration'),
    F.mean('member_attribution_stability').alias('avg_stability')
).collect()[0]

print("\nAggregate Metrics Across All Members:")
print(f"  Avg Risk-Adjusted Utilization Rate: {agg_stats['avg_util_rate']:.2f}")
print(f"  Avg Benefit Design Effectiveness: {agg_stats['avg_benefit_eff']:.2f}")
print(f"  Avg High-Value Care Penetration: {agg_stats['avg_hv_penetration']:.2f}")
print(f"  Avg Member Attribution Stability: {agg_stats['avg_stability']:.2f}")


FINAL METRICS SUMMARY
+---------+----------------+------------------------------------+----------------------------+---------------------------+----------------------------+--------------+---------------+---------------+
|person_id|payer_concept_id|risk_adjusted_utilization_rate_final|benefit_design_effectiveness|high_value_care_penetration|member_attribution_stability|hcc_risk_score|total_cost_pmpm|enrollment_days|
+---------+----------------+------------------------------------+----------------------------+---------------------------+----------------------------+--------------+---------------+---------------+
|477637   |0               |0.0                                 |70.0                        |0.0                        |39.0                        |1.0           |0.0            |0              |
|1542531  |0               |0.0                                 |70.0                        |0.0                        |39.0                        |1.0           |0.0            

In [51]:
metrics_csv = f"{LOCAL_DATA_DIR}/final_payer_metrics.csv"
final_metrics.toPandas().to_csv(metrics_csv, index=False)
print(f"\nSaved final metrics to: {metrics_csv}")


Saved final metrics to: ./5_data/final_payer_metrics.csv


# STEP 6: DAG Construction

In [52]:
print("\n" + "="*80)
print("DAG CONSTRUCTION")
print("="*80)

G = nx.DiGraph()
print("Initialized directed graph")


DAG CONSTRUCTION
Initialized directed graph


In [53]:
print("\nAdding Bronze Layer Nodes...")

bronze_nodes = [
    ('bronze.person', 'Person demographics'),
    ('bronze.observation_period', 'Observation periods'),
    ('bronze.payer_plan_period', 'Insurance coverage periods'),
    ('bronze.condition_occurrence', 'Diagnosis records'),
    ('bronze.procedure_occurrence', 'Procedure records'),
    ('bronze.drug_exposure', 'Drug prescriptions'),
    ('bronze.cost', 'Cost records'),
    ('bronze.care_site', 'Facility information'),
    ('bronze.provider', 'Provider information'),
    ('bronze.inpatient_charges_2011', 'Medicare inpatient charges')
]

for node_id, label in bronze_nodes:
    G.add_node(node_id, id=node_id, label=label, layer='bronze')

print(f"Added {len(bronze_nodes)} Bronze nodes")


Adding Bronze Layer Nodes...
Added 10 Bronze nodes


In [54]:
print("\nAdding Silver Layer Nodes...")

silver_nodes = [
    ('silver.member_enrollment', 'Member demographics & enrollment'),
    ('silver.utilization', 'Service utilization aggregation'),
    ('silver.cost_by_payer', 'Cost attribution by payer'),
    ('silver.provider_attribution', 'Provider network attribution'),
    ('silver.risk_scores', 'HCC risk score calculation'),
    ('silver.plan_features', 'Plan benefit design features')
]

for node_id, label in silver_nodes:
    G.add_node(node_id, id=node_id, label=label, layer='silver')

print(f"Added {len(silver_nodes)} Silver nodes")


Adding Silver Layer Nodes...
Added 6 Silver nodes


In [55]:
print("\nAdding Gold Layer Nodes (15 levels)...")

gold_nodes = [
    ('gold.l1_member_profile', 'L1: Integrated member profile'),
    ('gold.l2_utilization_density', 'L2: Utilization density metrics'),
    ('gold.l3_risk_adjusted', 'L3: Risk-adjusted utilization'),
    ('gold.l4_pmpm', 'L4: Cost per member per month'),
    ('gold.l5_risk_adj_pmpm', 'L5: Risk-adjusted PMPM'),
    ('gold.l6_coverage', 'L6: Plan coverage effectiveness'),
    ('gold.l7_high_value', 'L7: High-value service utilization'),
    ('gold.l8_attribution', 'L8: Provider attribution continuity'),
    ('gold.l9_stability', 'L9: Member tenure & stability'),
    ('gold.l10_benchmarked', 'L10: Plan performance benchmarking'),
    ('gold.l11_vbc', 'L11: Value-based care alignment'),
    ('gold.l12_engagement', 'L12: Member engagement index'),
    ('gold.l13_metric1', 'L13: METRIC 1 - Risk-Adjusted Utilization Rate'),
    ('gold.l14_metric2', 'L14: METRIC 2 - Benefit Design Effectiveness'),
    ('gold.l15_metrics_34', 'L15: METRICS 3 & 4 - HV Penetration + Attribution Stability')
]

for node_id, label in gold_nodes:
    G.add_node(node_id, id=node_id, label=label, layer='gold')

print(f"Added {len(gold_nodes)} Gold nodes")


Adding Gold Layer Nodes (15 levels)...
Added 15 Gold nodes


In [56]:
print("\nAdding Edges (Data Dependencies)...")

bronze_to_silver = [
    ('bronze.person', 'silver.member_enrollment'),
    ('bronze.observation_period', 'silver.member_enrollment'),
    ('bronze.payer_plan_period', 'silver.member_enrollment'),
    ('bronze.condition_occurrence', 'silver.utilization'),
    ('bronze.procedure_occurrence', 'silver.utilization'),
    ('bronze.drug_exposure', 'silver.utilization'),
    ('bronze.cost', 'silver.cost_by_payer'),
    ('bronze.payer_plan_period', 'silver.cost_by_payer'),
    ('bronze.condition_occurrence', 'silver.provider_attribution'),
    ('bronze.provider', 'silver.provider_attribution'),
    ('bronze.payer_plan_period', 'silver.plan_features')
]

silver_to_gold = [
    ('silver.member_enrollment', 'gold.l1_member_profile'),
    ('silver.utilization', 'gold.l1_member_profile'),
    ('silver.cost_by_payer', 'gold.l1_member_profile'),
    ('silver.provider_attribution', 'gold.l1_member_profile'),
    ('silver.risk_scores', 'gold.l1_member_profile')
]

gold_vertical = [
    ('gold.l1_member_profile', 'gold.l2_utilization_density'),
    ('gold.l2_utilization_density', 'gold.l3_risk_adjusted'),
    ('gold.l3_risk_adjusted', 'gold.l4_pmpm'),
    ('gold.l4_pmpm', 'gold.l5_risk_adj_pmpm'),
    ('gold.l5_risk_adj_pmpm', 'gold.l6_coverage'),
    ('gold.l6_coverage', 'gold.l7_high_value'),
    ('gold.l7_high_value', 'gold.l8_attribution'),
    ('gold.l8_attribution', 'gold.l9_stability'),
    ('gold.l9_stability', 'gold.l10_benchmarked'),
    ('gold.l10_benchmarked', 'gold.l11_vbc'),
    ('gold.l11_vbc', 'gold.l12_engagement'),
    ('gold.l12_engagement', 'gold.l13_metric1'),
    ('gold.l13_metric1', 'gold.l14_metric2'),
    ('gold.l14_metric2', 'gold.l15_metrics_34')
]

additional_edges = [
    ('silver.plan_features', 'gold.l6_coverage')
]

all_edges = bronze_to_silver + silver_to_gold + gold_vertical + additional_edges
for src, dst in all_edges:
    G.add_edge(src, dst, etype='consume')

print(f"Added {len(all_edges)} edges")
print(f"  Bronze -> Silver: {len(bronze_to_silver)}")
print(f"  Silver -> Gold: {len(silver_to_gold)}")
print(f"  Gold vertical: {len(gold_vertical)}")
print(f"  Additional: {len(additional_edges)}")


Adding Edges (Data Dependencies)...
Added 31 edges
  Bronze -> Silver: 11
  Silver -> Gold: 5
  Gold vertical: 14
  Additional: 1


In [57]:
print("\n" + "="*80)
print("GENERATE RAG DATA")
print("="*80)

rag_data = []
for node_id in G.nodes():
    node_data = G.nodes[node_id]
    predecessors = list(G.predecessors(node_id))
    successors = list(G.successors(node_id))
    
    rag_entry = {
        'id': node_id,
        'label': node_data.get('label', ''),
        'layer': node_data.get('layer', ''),
        'predecessors': predecessors,
        'successors': successors,
        'in_degree': len(predecessors),
        'out_degree': len(successors)
    }
    rag_data.append(rag_entry)

print(f"Generated RAG data for {len(rag_data)} nodes")


GENERATE RAG DATA
Generated RAG data for 31 nodes


In [58]:
print("\n" + "="*80)
print("SAVE DAG & RAG DATA")
print("="*80)

dag_file = f"{LOCAL_DATA_DIR}/payer_utilization_dag.graphml"
nx.write_graphml(G, dag_file)
print(f"Saved DAG: {dag_file}")

rag_file = f"{LOCAL_DATA_DIR}/payer_utilization_rag_data.json"
with open(rag_file, 'w') as f:
    json.dump(rag_data, f, indent=2)
print(f"Saved RAG data: {rag_file}")

stats = {
    'total_nodes': G.number_of_nodes(),
    'total_edges': G.number_of_edges(),
    'bronze_nodes': len(bronze_nodes),
    'silver_nodes': len(silver_nodes),
    'gold_nodes': len(gold_nodes),
    'is_dag': nx.is_directed_acyclic_graph(G),
    'timestamp': datetime.now().isoformat()
}

stats_file = f"{LOCAL_DATA_DIR}/dag_statistics.json"
with open(stats_file, 'w') as f:
    json.dump(stats, f, indent=2)
print(f"Saved statistics: {stats_file}")

print("\n" + "="*80)
print("DAG CONSTRUCTION COMPLETE")
print(f"  Nodes: {G.number_of_nodes()} ({len(bronze_nodes)} Bronze + {len(silver_nodes)} Silver + {len(gold_nodes)} Gold)")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Files: 3 (DAG + RAG + Stats)")
print("="*80)


SAVE DAG & RAG DATA
Saved DAG: ./5_data/payer_utilization_dag.graphml
Saved RAG data: ./5_data/payer_utilization_rag_data.json
Saved statistics: ./5_data/dag_statistics.json

DAG CONSTRUCTION COMPLETE
  Nodes: 31 (10 Bronze + 6 Silver + 15 Gold)
  Edges: 31
  Files: 3 (DAG + RAG + Stats)


In [59]:
print("\n" + "="*80)
print("DAG VALIDATION")
print("="*80)

print(f"\nTotal Nodes: {G.number_of_nodes()}")
print(f"Total Edges: {G.number_of_edges()}")
print(f"\nIs DAG: {nx.is_directed_acyclic_graph(G)}")
print(f"Is Weakly Connected: {nx.is_weakly_connected(G)}")

sources = [n for n in G.nodes() if G.in_degree(n) == 0]
print(f"\nSource Nodes (no incoming): {len(sources)}")
for s in sources[:5]:
    print(f"  - {s}")
if len(sources) > 5:
    print(f"  ... and {len(sources)-5} more")

sinks = [n for n in G.nodes() if G.out_degree(n) == 0]
print(f"\nSink Nodes (no outgoing): {len(sinks)}")
for s in sinks:
    print(f"  - {s}")

print("\n" + "="*80)
print("PIPELINE COMPLETE")
print("="*80)


DAG VALIDATION

Total Nodes: 31
Total Edges: 31

Is DAG: True
Is Weakly Connected: False

Source Nodes (no incoming): 11
  - bronze.person
  - bronze.observation_period
  - bronze.payer_plan_period
  - bronze.condition_occurrence
  - bronze.procedure_occurrence
  ... and 6 more

Sink Nodes (no outgoing): 3
  - bronze.care_site
  - bronze.inpatient_charges_2011
  - gold.l15_metrics_34

PIPELINE COMPLETE


In [60]:
# Verify Lineage
print("\n" + "="*60)
print("LINEAGE TRACKING COMPLETE!")
print("="*60)
print(f"✅ Notebook completed successfully")
print(f"✅ Marquez Web UI: http://localhost:3601")
print(f"✅ Namespace: payer_utilization")  # CORRECTED!
print(f"\nNext Steps:")
print("1. Open http://localhost:3601 in your browser")
print("2. Select namespace: 'payer_utilization'")
print("3. Browse Jobs and Datasets")
print("4. Click on any dataset to see lineage graph")
print("="*60)

# CRITICAL: Stop Spark to send job completion event
print("\nStopping Spark session to complete job...")
spark.stop()
print("✅ Spark session stopped - job marked as COMPLETE in Marquez")


LINEAGE TRACKING COMPLETE!
✅ Notebook completed successfully
✅ Marquez Web UI: http://localhost:3601
✅ Namespace: payer_utilization

Next Steps:
1. Open http://localhost:3601 in your browser
2. Select namespace: 'payer_utilization'
3. Browse Jobs and Datasets
4. Click on any dataset to see lineage graph

Stopping Spark session to complete job...


25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 88
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 88
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 88
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 89
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 89
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 90
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 90
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 90
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for executionId 91
25/11/23 23:07:59 ERROR ContextFactory: Query execution is null: can't emit event for execu

✅ Spark session stopped - job marked as COMPLETE in Marquez
